# OFDM BER Simulator

Este notebook implementa un **simulador interactivo de un enlace digital OFDM** para estudiar cómo cambian la **constelación recibida**, la **BER** y el efecto de la **ecualización** y del **FEC** cuando se modifica la **SNR** del canal.

El modelo representa la cadena completa **Tx-Canal-Rx**:

1. Generación de bits aleatorios  
2. Codificación FEC opcional  
3. Modulación M-QAM  
4. Mapeo OFDM sobre `Nsub` subportadoras  
5. IFFT y adición del prefijo cíclico `CP`  
6. Transmisión por un canal `AWGN` o `Rayleigh`  
7. FFT en recepción  
8. Ecualización opcional en frecuencia  
9. Demodulación QAM  
10. Decodificación FEC opcional  
11. Cálculo de BER y representación gráfica

El objetivo didáctico es comparar tres situaciones:

- **Sin ecualización y sin FEC**
- **Con ecualización y sin FEC**
- **Con ecualización y con FEC**

y observar cómo cambia el rendimiento frente a la SNR.

---

## Parámetros del simulador

### Parámetros OFDM

**Nsub**  
Número de subportadoras OFDM. Define el tamaño de la IFFT/FFT y el número de símbolos modulados en paralelo.

**CP Length**  
Longitud del prefijo cíclico. Si el canal es dispersivo, el CP ayuda a evitar interferencia entre símbolos siempre que sea suficientemente largo respecto a la dispersión temporal del canal.

---

### Parámetros de modulación

**Mod Order (M-QAM)**  
Orden de la modulación por subportadora. En el simulador se trabaja con `4`, `16` y `64`. A mayor orden, más bits por símbolo, pero también mayor sensibilidad al ruido y a la distorsión.

---

### Parámetros de canal

**SNR min / SNR max**  
Rango de SNR sobre el que se barre para construir la curva `BER vs SNR`.

**Channel type**  
Tipo de canal:

- `AWGN`: solo ruido blanco gaussiano
- `Rayleigh`: canal dispersivo con multitrayectoria

**n_taps**  
Número de trayectorias del canal Rayleigh. Más taps implican un canal más selectivo en frecuencia y, por tanto, mayor necesidad de ecualización y de un CP adecuado.

---

### Parámetros de simulación

**Frames**  
Número de tramas OFDM utilizadas para estimar la BER en cada punto de la curva. Más tramas implican una estimación más estable, pero también más tiempo de simulación.

**FEC ON**  
Activa la codificación de canal. El notebook permite comparar el caso sin FEC con el caso codificado para visualizar el desplazamiento del umbral de funcionamiento.

**FEC n / FEC k**  
Definen la tasa de código del FEC sencillo implementado en el simulador. A menor tasa, mayor redundancia y mayor robustez, a costa de transmitir menos información útil por símbolo.

**Ecualización**  
En canal Rayleigh, el receptor puede comparar la señal antes y después de ecualizar, lo que permite observar la mejora en la constelación y en la BER.

---

## Salidas del notebook

Tras ejecutar la simulación, el notebook muestra:

- constelación transmitida y recibida
- constelación recibida antes y después de ecualizar
- espectro recibido antes y después de ecualizar
- curvas `BER vs SNR` para las distintas configuraciones del receptor

Estas gráficas permiten relacionar de forma visual el efecto del ruido, la selectividad del canal, la ecualización y el FEC con el rendimiento final del sistema.

---

## Uso básico

1. Ajustar los parámetros del sistema  
2. Ejecutar la simulación desde el panel  
3. Comparar constelaciones, espectros y curvas BER obtenidas


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
package_root = project_root / 'Engine'
helper_root = project_root / 'Helpers'

for path in [package_root, helper_root]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print('Project root:', project_root)
print('Active package root:', package_root)
print('Helper root:', helper_root)


In [ ]:
from SNR_BER_helper import *



## Notas

- El panel replica la estructura visual del GUI de MATLAB: **Simulación**, **Parámetros Tx**, **Parámetros Canal**, **Ejecución** y **Resultados**.
- El motor sigue siendo el de tus módulos Python.
- El dropdown **Modulation** se deja en `QAM`, porque el código actual solo soporta 4/16/64-QAM.
- Internamente se mantienen `channel_type` y `n_taps`; si quieres que también aparezcan en el panel visual, se pueden añadir sin tocar el motor.


## Preguntas

1. ¿Qué ocurre con la **BER** cuando aumenta la **SNR**?

2. Compare las curvas para **QPSK**, **16-QAM** y **64-QAM**.  
   ¿Qué modulación necesita mayor SNR para obtener una BER baja?

3. Observe la constelación recibida cuando la SNR es baja y cuando es alta.  
   ¿Cómo cambia la dispersión de los símbolos?

4. Compare la recepción **sin ecualización** y **con ecualización** en canal dispersivo.  
   ¿Qué efecto tiene la ecualización sobre la constelación y sobre la BER?

5. Active el **FEC** y compare la curva BER frente al caso sin FEC.  
   ¿Qué significa que el FEC desplace el umbral de funcionamiento?

6. Si se aumenta el número de subportadoras **Nsub**, ¿qué efecto esperaría sobre la duración del símbolo OFDM y sobre la sensibilidad a la dispersión temporal?

7. Explique por qué una constelación de orden alto, como **64-QAM**, transmite más bits por símbolo pero también resulta más sensible al ruido.


<details>
<summary>Ver soluciones</summary>

### 1

Al aumentar la SNR, el ruido relativo disminuye y la BER tiende a reducirse.

---

### 2

Las modulaciones de mayor orden, como 64-QAM, necesitan una SNR más alta para mantener una BER baja.

---

### 3

Con SNR baja, la nube de puntos aparece más dispersa. Con SNR alta, los símbolos se concentran alrededor de los puntos ideales.

---

### 4

La ecualización compensa la distorsión introducida por el canal dispersivo y suele reducir la BER, además de hacer la constelación más reconocible.

---

### 5

El FEC permite operar correctamente a SNR más baja. Ese desplazamiento del umbral significa que el sistema mantiene buen rendimiento en condiciones más exigentes.

---

### 6

Al aumentar Nsub, el símbolo OFDM se hace más largo en el tiempo y cada subportadora ocupa menos ancho de banda. Eso suele ayudar frente a canales selectivos en frecuencia, aunque también afecta a la configuración del CP.

---

### 7

64-QAM transmite más bits por símbolo porque tiene más puntos de constelación, pero esos puntos están más próximos entre sí y son más fáciles de confundir cuando hay ruido o distorsión.
</details>

---
